# Phase 3: Feature Engineering & Data Management

This notebook implements the engineering of clinical features (BMI, Pulse Pressure, and an Age-Cholesterol Risk Interaction feature), performs feature selection using statistical correlation and Mutual Information, stores the processed dataset in a local SQLite database, and executes 4 validation queries (including the 3 custom queries designed by our team).

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from sklearn.feature_selection import mutual_info_classif

In [ ]:
df = pd.read_csv('../data/cardio_train.csv', sep=';')
df_clean = df[
    (df['ap_hi'] >= 80) & (df['ap_hi'] <= 220) &
    (df['ap_lo'] >= 50) & (df['ap_lo'] <= 140) &
    (df['ap_hi'] > df['ap_lo']) &
    (df['height'] >= 130) & (df['height'] <= 220) &
    (df['weight'] >= 40) & (df['weight'] <= 180)
].copy()
df_clean['gender'] = df_clean['gender'] - 1
df_clean.shape

## 1. Feature Engineering

We derive 3 custom clinical indicators:
1. **Body Mass Index (BMI):** $\text{weight (kg)} / (\text{height (m)})^2$
2. **Pulse Pressure:** $\text{ap\_hi} - \text{ap\_lo}$ (difference between systolic and diastolic blood pressure, representing arterial stiffness)
3. **Age-Cholesterol Risk Interaction:** $(\text{age in years}) \times \text{cholesterol}$ (compounding risk of arterial stiffness/aging with lipid levels)

In [ ]:
df_clean['bmi'] = df_clean['weight'] / ((df_clean['height'] / 100) ** 2)
df_clean['pulse_pressure'] = df_clean['ap_hi'] - df_clean['ap_lo']
df_clean['age_cholesterol_risk'] = (df_clean['age'] / 365.25) * df_clean['cholesterol']
df_clean.head()

## 2. Feature Selection Report

We calculate the Mutual Information (MI) classification scores for all features relative to the target variable `cardio` to rank their predictive power.

In [ ]:
X = df_clean.drop(columns=['id', 'cardio'])
y = df_clean['cardio']
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_df = pd.DataFrame({'Feature': X.columns, 'MI Score': mi_scores})
mi_df = mi_df.sort_values(by='MI Score', ascending=False).reset_index(drop=True)
mi_df

### Feature Selection Decisions

**Features Kept:**
- `pulse_pressure`, `ap_hi`, `ap_lo` (very high correlation/MI with target; represents the hemodynamic state).
- `age`, `age_cholesterol_risk`, `cholesterol` (age and lipid levels are key demographic and metabolic risk factors).
- `bmi`, `weight`, `height` (vital markers for physical build, obesity, and systemic strain).
- `gluc`, `smoke`, `alco`, `active`, `gender` (subjective habits and clinical indicators that represent distinct paths to cardiovascular risk).

**Features Dropped:**
- `id`: Removed because it is a randomly assigned integer identifier and carries no predictive value or biological relevance.

Our newly engineered feature, `pulse_pressure`, ranks high in mutual information importance, confirming our EDA finding that the difference between systolic and diastolic pressure creates a clearer decision boundary than other features, particularly in sub-55 patient cohorts.

## 3. Data Management (SQLite Integration)

We initialize a local SQLite instance at `../data/cardio_cleaned.db`, store the cleaned and engineered dataset in a table named `cardio`, and execute our 4 validation queries.

In [ ]:
db_path = '../data/cardio_cleaned.db'
conn = sqlite3.connect(db_path)
df_clean.to_sql('cardio', conn, if_exists='replace', index=False)
cursor = conn.cursor()

### Query 1 (Custom)

**Goal:** Calculate the percentage of patients with cardiovascular disease (`cardio = 1`) grouped by physical activity level (`active`) and alcohol consumption (`alco`).

In [ ]:
q1 = """
SELECT 
    active, 
    alco, 
    AVG(cardio) * 100 AS cvd_percentage, 
    COUNT(*) AS patient_count
FROM cardio 
GROUP BY active, alco;
"""
pd.read_sql_query(q1, conn)

### Query 2 (Custom)

**Goal:** Analyze the average `pulse_pressure` and average `bmi` grouped by cholesterol level (1, 2, and 3), filtered for patients older than 50 (converting age from days to years using 365.25).

In [ ]:
q2 = """
SELECT 
    cholesterol, 
    AVG(pulse_pressure) AS avg_pulse_pressure, 
    AVG(bmi) AS avg_bmi, 
    COUNT(*) AS patient_count
FROM cardio 
WHERE (age / 365.25) > 50 
GROUP BY cholesterol;
"""
pd.read_sql_query(q2, conn)

### Query 3 (Custom)

**Goal:** Count the number of high-risk patients without a cardiovascular diagnosis (`cardio = 0`) who have high cholesterol (3), high glucose (3), and are obese (`bmi > 30`).

In [ ]:
q3 = """
SELECT COUNT(*) AS high_risk_no_cvd_count 
FROM cardio 
WHERE cardio = 0 
  AND cholesterol = 3 
  AND gluc = 3 
  AND bmi > 30;
"""
pd.read_sql_query(q3, conn)

### Query 4 (Baseline Validation)

**Goal:** Perform basic validation showing overall counts, mean age in years, and mean BMI grouped by the target class (`cardio`).

In [ ]:
q4 = """
SELECT 
    cardio, 
    COUNT(*) AS total_patients, 
    AVG(age / 365.25) AS avg_age_years, 
    AVG(bmi) AS avg_bmi
FROM cardio 
GROUP BY cardio;
"""
pd.read_sql_query(q4, conn)

In [ ]:
conn.close()